In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import Select
import pandas as pd
import datetime
from datetime import date
import random
import subprocess
import time as t
import csv
import re
import os

link = "https://www.southernrailway.com"

leaving_from = "Clapham Junction"
going_to = "Brighton"

In [ ]:



driver = webdriver.Edge()
driver.get(link)
t.sleep(5)

button = driver.find_element(By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
button.click()
t.sleep(3)

shadow_host = driver.find_element(By.ID, "otrl-custom-hero")
shadow_root = shadow_host.shadow_root
leaving =   shadow_root.find_element(By.NAME, "stationFrom")
leaving.send_keys(leaving_from)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()

t.sleep(3)
going = shadow_root.find_element(By.NAME, "stationTo")
going.send_keys(going_to)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()


t.sleep(3)

search = shadow_root.find_element(By.CLASS_NAME, "otrl-jp__tickets__submit")
search.click()
t.sleep(5)

listview = driver.find_element(By.CSS_SELECTOR, 'a[aria-label="List view"]')
listview.click()
t.sleep(5)

wait = WebDriverWait(driver, 2)
seen_departures = set()
records = []

try:
    date_text = driver.find_element(By.CSS_SELECTOR, ".service-list__heading2").text.strip()  # e.g. "Wed 04 Mar 2026"
    current_date = datetime.datetime.strptime(date_text, "%a %d %b %Y").date()
except Exception as e:
    print(f"  Could not parse date: {e}")

current_date_tracker = current_date


def parse_current_page(current_date_tracker):
    """Parse all visible trains and fares from list view, skip already-seen departures."""

    cards = driver.find_elements(By.CSS_SELECTOR, ".service-list__card.service-fare")

    for i, card in enumerate(cards):
        try:
            # Journey info from sr-text h4
            sr_text = card.find_element(By.CSS_SELECTOR, "h4.sr-text").text

            dep_match = re.search(r'^(\d{2}:\d{2})', sr_text)
            arr_match = re.search(r'arriving at .+? at (\d{2}:\d{2})', sr_text)
            dur_match = re.search(r'takes (.+?),', sr_text)
            chg_match = re.search(r'has (\d+) change', sr_text)

            departure = dep_match.group(1) if dep_match else None
            if not departure:
                continue

            arrival  = arr_match.group(1) if arr_match else None
            dep_time = datetime.datetime.strptime(departure, "%H:%M").time()
            current_date = current_date_tracker

            if records:
                last_dep_dt = records[-1]["departure_dt"]
                candidate_dt = datetime.datetime.combine(current_date, dep_time)
                if (last_dep_dt - candidate_dt).total_seconds() > 6 * 3600:
                    current_date = current_date + datetime.timedelta(days=1)
                    current_date_tracker = current_date
                    print(f"  Date rolled over to {current_date}")

            dedup_key = (current_date, departure)
            if dedup_key in seen_departures:
                continue

            duration = dur_match.group(1).strip() if dur_match else None
            changes  = int(chg_match.group(1)) if chg_match else 0

            # Price from the continue button — contained within the same card
            price = None
            try:
    # Target the sr-text span inside the button for a clean price string
                price_element = card.find_element(By.CSS_SELECTOR, ".btn-continue .sr-text")
                price_text = price_element.text  
    
    # Clean the string (remove £) and convert to float
                price = float(price_text.replace('£', '').strip())
            except Exception:
                pass  # No fare available for this service

            departure_dt = datetime.datetime.combine(current_date, dep_time)
            arrival_dt   = None
            if arrival:
                arr_time = datetime.datetime.strptime(arrival, "%H:%M").time()
                arrival_dt = datetime.datetime.combine(current_date, arr_time)
                if arrival_dt < departure_dt:
                    arrival_dt += datetime.timedelta(days=1)

            records.append({
                "departure_dt": departure_dt,
                "arrival_dt":   arrival_dt,
                "departure":    departure,
                "arrival":      arrival,
                "duration":     duration,
                "changes":      changes,
                "price_gbp":    price,
            })
            seen_departures.add(dedup_key)

        except Exception as e:
            print(f"  Error parsing traincard {i}: {e}")


parse_current_page(current_date_tracker)


for click_num in range(15):
    try:
        t.sleep(random.uniform(0.3, 1.4))

        time_elements = driver.find_elements(
            By.CSS_SELECTOR, ".service-list-v2__services li .service-summary__station time"
        )
        if not time_elements:
            print(f"Click {click_num + 1}: no time elements found before click, skipping.")
            break
        last_departure_before = time_elements[-1].get_attribute("datetime")

        later_btn = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "a.service-pager[aria-label='Show later trains']")
        ))
        driver.execute_script("arguments[0].click();", later_btn)

        def page_has_updated(d):
            try:
                elements = d.find_elements(
                    By.CSS_SELECTOR, ".service-list-v2__services li .service-summary__station time"
                )
                return (
                    len(elements) > 0
                    and elements[-1].get_attribute("datetime") != last_departure_before
                )
            except Exception:
                return False

        WebDriverWait(driver, 10).until(page_has_updated)

        # Wait for fare buttons to be rendered before scraping prices
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".service-list-v2__services li .btn-continue"))
        )

    except TimeoutException:
        print(f"Click {click_num + 1}: page didn't update or no 'Later' button.")
        break

    parse_current_page(current_date_tracker)
    print(f"After click {click_num + 1}: {len(records)} trains total")


driver.quit()
df = pd.DataFrame(records)
print(df.head())

In [21]:
#attempt to enter date range and re-scrape - not working yet

driver = webdriver.Edge()
driver.get(link)
t.sleep(5)

button = driver.find_element(By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
button.click()
t.sleep(3)

shadow_host = driver.find_element(By.ID, "otrl-custom-hero")
shadow_root = shadow_host.shadow_root
leaving =   shadow_root.find_element(By.NAME, "stationFrom")
leaving.send_keys(leaving_from)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()

t.sleep(3)

going = shadow_root.find_element(By.NAME, "stationTo")
going.send_keys(going_to)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()


t.sleep(3)

select_outbound_date_time(driver, 15, "April 2026", "10:00")


search = shadow_root.find_element(By.CLASS_NAME, "otrl-jp__tickets__submit")
search.click()
t.sleep(5)


Date set to: 15 April 2026 at 10:0


In [20]:
def select_outbound_date_time(driver, day: int, month_year: str, time_str: str = "Now"):
    """
    Example usage:
        select_outbound_date_time(driver, 15, "April 2026", "10:00")
    """
    wait = WebDriverWait(driver, 10)

    # 1. Pierce shadow DOM
    shadow_host = wait.until(
        EC.presence_of_element_located((By.ID, "otrl-custom-hero"))
    )
    shadow_root = driver.execute_script(
        "return arguments[0].shadowRoot", shadow_host
    )

    # 2. Open date picker
    date_input = shadow_root.find_element(
        By.CSS_SELECTOR, ".otrl-jp__date-input"
    )
    date_input.click()
    t.sleep(1)

    # 3. Navigate to correct month
    for _ in range(12):
        current = shadow_root.find_element(
            By.CSS_SELECTOR, ".DayPicker-Caption div"
        ).text
        if current == month_year:
            break
        shadow_root.find_element(
            By.CSS_SELECTOR,
            ".otrl-ui__date-picker__month-selector__button--next"
        ).click()
        t.sleep(0.5)
    else:
        raise ValueError(f"Could not navigate to {month_year}")

    # 4. Click the day
    days = shadow_root.find_elements(
        By.CSS_SELECTOR,
        ".DayPicker-Day:not(.DayPicker-Day--disabled)"
        ":not(.DayPicker-Day--outside)"
    )
    for d in days:
        if d.text.strip() == str(day):
            d.click()
            t.sleep(0.5)
            break

        # 5. Select time from dropdown
    if time_str != "Now":
        hour, minute = map(int, time_str.split(":"))
        if minute not in [0, 15, 30, 45]:
            raise ValueError(
                f"Minute must be 00, 15, 30 or 45. Got: {minute:02d}"
            )
        index = (hour * 4) + (minute // 15) + 1
    else:
        index = 0

    time_dropdown = shadow_root.find_element(
    By.CSS_SELECTOR,
    "select.otrl-jp__time-input--hour-minute"
    # could also use: "select[name='time']"
    )
    Select(time_dropdown).select_by_index(index)
    t.sleep(0.3)

    # 6. Confirm
    shadow_root.find_element(
        By.CSS_SELECTOR, ".otrl-jp__date-popup__button"
    ).click()
    t.sleep(0.5)
    print(f"Date set to: {day} {month_year} at {hour}:{minute}")


In [ ]:
# Click the date input button inside the shadow root
date_input = shadow_root.find_element(By.CSS_SELECTOR, ".otrl-jp__date-input")
date_input.click()
t.sleep(1)  # allow the calendar popup to animate open

def get_current_month_year(shadow_root):
    caption = shadow_root.find_element(
        By.CSS_SELECTOR, ".DayPicker-Caption div"
    )
    return caption.text  # e.g. "March 2026"

def click_next_month(shadow_root):
    next_btn = shadow_root.find_element(By.CSS_SELECTOR, ".otrl-ui__date-picker__month-selector__button--next")
    next_btn.click()
    t.sleep(0.5)

def click_prev_month(shadow_root):
    prev_btn = shadow_root.find_element(By.CSS_SELECTOR,".otrl-ui__date-picker__month-selector__button--prev")
    prev_btn.click()
    t.sleep(0.5)

# Navigate to a target month if needed
target_month = "April 2026"
for _ in range(6):  # safety limit
    current = get_current_month_year(shadow_root)
    if current == target_month:
        break
    click_next_month(shadow_root)

def select_day(shadow_root, day_number: int):
    # Get all available (non-disabled, non-outside) day cells
    days = shadow_root.find_elements(
        By.CSS_SELECTOR,
        ".DayPicker-Day:not(.DayPicker-Day--disabled)"
        ":not(.DayPicker-Day--outside)"
    )
    for day in days:
        if day.text.strip() == str(day_number):
            day.click()
            t.sleep(0.5)
            return
    raise ValueError(f"Day {day_number} not found or not available")

select_day(shadow_root, 15) 

def select_time(shadow_root, hour: str, minute: str = "00"):
    """
    hour   — e.g. "10", "14"
    minute — e.g. "00", "15", "30", "45"
    """
    time_selects = shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-ui__select"
    )
    
    # First select is typically hours, second is minutes
    if len(time_selects) >= 1:
        Select(time_selects[0]).select_by_value(hour)
    if len(time_selects) >= 2:
        Select(time_selects[1]).select_by_value(minute)
    
    t.sleep(0.3)

select_time(shadow_root, hour="10", minute="00")

confirm_btn = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__date-popup__button"
)
confirm_btn.click()
t.sleep(0.5)